# 04 - 逻辑回归与分类

## 学习目标
- 理解逻辑回归的原理
- 使用 LogisticRegression 进行多分类
- 掌握分类任务的评估指标

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_curve, auc
)

## 1. 二分类：乳腺癌检测

In [ ]:
# 加载数据
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

print(f'数据集: {X.shape}')
print(f'类别分布: {np.bincount(y)} (0=恶性, 1=良性)')

# 划分与标准化
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# 训练模型
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

# 预测
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

# 评估
print('分类报告:')
print(classification_report(y_test, y_pred, target_names=data.target_names))

In [ ]:
# 混淆矩阵
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=data.target_names,
            yticklabels=data.target_names)
plt.xlabel('预测标签')
plt.ylabel('真实标签')
plt.title('混淆矩阵')
plt.show()

In [ ]:
# ROC 曲线
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc='lower right')
plt.show()

## 2. 多分类：鸢尾花

In [ ]:
# 加载鸢尾花数据
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.3, random_state=42, stratify=iris.target
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 多分类逻辑回归 (One-vs-Rest)
model = LogisticRegression(multi_class='multinomial', random_state=42, max_iter=200)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
print('多分类报告:')
print(classification_report(y_test, y_pred, target_names=iris.target_names))

## 3. 正则化参数 C 的影响

In [ ]:
# C 越大，正则化越弱（越复杂）
C_values = [0.001, 0.01, 0.1, 1, 10, 100]
results = []

for C in C_values:
    model = LogisticRegression(C=C, random_state=42, max_iter=1000)
    model.fit(X_train_scaled, y_train)
    train_acc = model.score(X_train_scaled, y_train)
    test_acc = model.score(X_test_scaled, y_test)
    results.append({'C': C, 'Train Acc': train_acc, 'Test Acc': test_acc})

results_df = pd.DataFrame(results)
print(results_df.round(4))

plt.figure(figsize=(10, 5))
plt.plot(results_df['C'], results_df['Train Acc'], 'o-', label='训练集')
plt.plot(results_df['C'], results_df['Test Acc'], 's-', label='测试集')
plt.xscale('log')
plt.xlabel('C (正则化强度的倒数)')
plt.ylabel('准确率')
plt.title('正则化参数 C 对模型性能的影响')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 小结
- 逻辑回归可用于二分类和多分类
- predict_proba 输出概率，用于 ROC 分析
- C 参数控制正则化强度

---
下一个: [05_decision_trees.ipynb](05_decision_trees.ipynb)